# Test 2: modelos locales

[Guía de la entrega](../../../../docs/README.md) · [Caso e historial](../README.md)

**Control manual:** 4B resolvió 7/12 casos; 8B, 6/12. Ambos fallaron validación. El negociador falla incluso sin mejorador.

Búsqueda fija/adaptativa v5; control manual en la sección 7. Lectura de evidencia guardada, sin llamadas al modelo.


In [1]:
from pathlib import Path
import hashlib
import html
import json
from IPython.display import HTML, display

relative = Path("research/cases/case-0003-debt-negotiation")
case_dir = next(base / relative for base in [Path.cwd(), *Path.cwd().parents]
                if (base / relative / "results").is_dir())
run_names = [
    "20261031T054930Z-local-qwen3-decision-fixed",
    "20261031T055008Z-local-qwen3-decision-adaptive",
    "20261031T055055Z-local-qwen3-8b-decision-fixed",
    "20261031T055221Z-local-qwen3-8b-decision-adaptive",
]

def load_record(name):
    folder = case_dir / "results" / name
    manifest = json.loads((folder / "manifest.json").read_text())
    for filename, expected in manifest.items():
        assert Path(filename).name == filename
        assert hashlib.sha256((folder / filename).read_bytes()).hexdigest() == expected
    return {"name": name,
        "protocol": json.loads((folder / "protocol.json").read_text()),
        "report": json.loads((folder / "report.json").read_text()),
        "calls": [json.loads(line) for line in (folder / "calls.jsonl").read_text().splitlines()]}

records = [load_record(name) for name in run_names]

def label(record):
    size = "8B" if record["report"]["model_key"] == "qwen3-8b" else "4B"
    mode = "B1 fija" if record["protocol"]["strategy_mode"] == "fixed" else "B2 adaptativa"
    return f"{size} · {mode}"

def table(headers, rows):
    head = "".join(f'<th style="text-align:left;padding:8px;border-bottom:2px solid #667085">{html.escape(str(h))}</th>' for h in headers)
    body = "".join("<tr>" + "".join(f'<td style="text-align:left;vertical-align:top;padding:8px;border-bottom:1px solid #ddd">{html.escape(str(v))}</td>' for v in row) + "</tr>" for row in rows)
    display(HTML(f'<table style="width:100%;border-collapse:collapse;margin:16px 0"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

def panel(title, text):
    display(HTML(f'<h4>{html.escape(title)}</h4><blockquote style="border-left:3px solid #667085;padding:12px;white-space:pre-wrap;overflow-wrap:anywhere">{html.escape(text)}</blockquote>'))

print("Integridad de las cuatro ejecuciones verificada.")
reference_names = [
    "20261031T170332Z-local-qwen3-reference",
    "20261031T170400Z-local-qwen3-8b-reference",
]
reference_records = [load_record(name) for name in reference_names]
assert reference_records[0]["protocol"]["reference_policy"] == reference_records[1]["protocol"]["reference_policy"]
print("Integridad de las dos ejecuciones de referencia verificada.")


Integridad de las cuatro ejecuciones verificada.
Integridad de las dos ejecuciones de referencia verificada.


## 1. Condiciones de ejecución

- Mac M5 Max, 48 GB de memoria unificada. MLX, pesos a 4 bits, sin adaptadores ni entrenamiento.
- Temperatura 0, semilla 42 y `enable_thinking=False`. Un modelo por proceso.
- Tres propuestas por condición; máximo 35 llamadas, 50.000 tokens, 15 minutos y 12 GiB de asignación MLX.
- Máximo 1.024 tokens de salida; entrada más salida reservada: 4.096 tokens.
- Dos casos de desarrollo, dos de validación y dos de test ya conocidos.
- Seis casos nuevos preparados antes de las cuatro ejecuciones, consultados solo después de seleccionar la política. No se usan para proponer ni elegir candidatos.

Las cuatro condiciones comparten el protocolo v5. Cambian el tamaño del modelo y la actualización de las instrucciones de mejora. El presupuesto máximo es común; el consumo real puede ser distinto.

## 2. Variables y controles

**H:** política. **φ:** instrucciones del mejorador. **L:** modelo de pesos fijos.

$H'_t=M_{\phi_t}(H_t,O,F_{dev},historial;L)$.

El mejorador recibe objetivo O, fallos de desarrollo y dos intentos recientes. **B1:** φ fija. **B2:** otra llamada actualiza φ desde la candidata y sus fallos. Evaluador Python fijo.

**Flujo:** H + caso → LLM → decisión JSON → plantilla → evaluación.

El modelo devuelve `action`, `debt`, `payments` e `installment`; Python redacta `message`. No corrige decisiones ni montos.


In [2]:
table(["Parte", "Responsabilidad", "Editable por el modelo"], [
    ["Negociador", "Decidir la acción y el plan", "Política H"],
    ["Mejorador", "Proponer otra política usando errores e intentos recientes", "Instrucciones φ en B2"],
    ["Actualizador", "Proponer las instrucciones de la siguiente iteración", "Solo el texto φ"],
    ["Renderizador", "Redactar un mensaje desde los campos devueltos", "No"],
    ["Evaluador", "Comprobar decisión y restricciones; seleccionar", "No"],
])
panel("Contrato fijo del negociador", records[0]["protocol"]["runtime"])

Parte,Responsabilidad,Editable por el modelo
Negociador,Decidir la acción y el plan,Política H
Mejorador,Proponer otra política usando errores e intentos recientes,Instrucciones φ en B2
Actualizador,Proponer las instrucciones de la siguiente iteración,Solo el texto φ
Renderizador,Redactar un mensaje desde los campos devueltos,No
Evaluador,Comprobar decisión y restricciones; seleccionar,No


## 3. Resultado de búsqueda

**Q:** decisiones correctas sin infracciones / respuestas evaluadas. Aceptación: mejora de Q en validación, sin infracciones.

Duplicados: igualdad de texto normalizando Unicode y espacios. Reutilizan puntuaciones; no cuentan como mejoras. No se detectan paráfrasis.


In [3]:
quality_rows, resource_rows = [], []
for record in records:
    r = record["report"]
    selected = next(item for item in r["rounds"] if item["iteration"] == r["selected_iteration"])
    duplicates = sum(item.get("duplicate_of") is not None for item in r["rounds"][1:])
    quality_rows.append([label(record), len(r["rounds"]) - 1 - duplicates, duplicates,
        sum(item["accepted"] for item in r["rounds"][1:]),
        f'{selected["validation"]["q"] * 2:.0f}/2',
        f'{r["transfer"]["initial"]["q"] * 6:.0f}/6',
        f'{r["transfer"]["selected"]["q"] * 6:.0f}/6'])
    resource_rows.append([label(record), r["calls"], r["tokens"],
        f'{r["elapsed_seconds"]:.1f}', f'{r["peak_mlx_gb"]:.2f}'])
table(["Condición", "Propuestas nuevas", "Repetidas", "Aceptadas", "Validación", "Nuevos H0", "Nuevos Hfinal"], quality_rows)
table(["Condición", "Llamadas", "Tokens totales", "Segundos", "Pico MLX (GB)"], resource_rows)

Condición,Propuestas nuevas,Repetidas,Aceptadas,Validación,Nuevos H0,Nuevos Hfinal
4B · B1 fija,1,2,0,0/2,0/6,0/6
4B · B2 adaptativa,2,1,0,0/2,0/6,0/6
8B · B1 fija,1,2,0,0/2,2/6,2/6
8B · B2 adaptativa,1,2,0,0/2,2/6,2/6


Condición,Llamadas,Tokens totales,Segundos,Pico MLX (GB)
4B · B1 fija,19,11288,14.1,3.18
4B · B2 adaptativa,25,16098,19.4,3.22
8B · B1 fija,19,11578,42.2,5.61
8B · B2 adaptativa,21,13868,47.0,5.66


Las cuatro condiciones conservaron **H0**. El 2/6 de 8B corresponde a esa política inicial. Test reutilizado: 1/2 en las cuatro.

Pico MLX: unos 5,6 GB en 8B y 3,2 GB en 4B; no equivale a toda la RAM. Tiempos con carga, sin descarga. Una ejecución por condición, 4B antes de 8B.

La plantilla conserva los valores del modelo, incluidos errores.


## 4. Control de duplicados

El mejorador recibe dos intentos recientes y sus fallos. Un duplicado evita cuatro reevaluaciones: dos de desarrollo y dos de validación. Su generación sí consume una llamada.


In [4]:
rows = []
for record in records:
    for item in record["report"]["rounds"][1:]:
        rows.append([label(record), f'H{item["iteration"]}',
            f'H{item["parent_iteration"]}',
            "Nueva" if item["duplicate_of"] is None else f'Repite H{item["duplicate_of"]}',
            item["policy_tokens"], item["strategy_tokens"]])
table(["Condición", "Propuesta", "Desde", "Identidad", "Tokens H", "Tokens φ usados"], rows)
count = sum(item.get("duplicate_of") is not None for r in records for item in r["report"]["rounds"][1:])
print(f"{count} propuestas duplicadas detectadas; {4 * count} llamadas de evaluación omitidas.")

Condición,Propuesta,Desde,Identidad,Tokens H,Tokens φ usados
4B · B1 fija,H1,H0,Nueva,107,31
4B · B1 fija,H2,H0,Repite H1,107,31
4B · B1 fija,H3,H0,Repite H1,107,31
4B · B2 adaptativa,H1,H0,Nueva,107,31
4B · B2 adaptativa,H2,H0,Nueva,121,123
4B · B2 adaptativa,H3,H0,Repite H2,121,123
8B · B1 fija,H1,H0,Nueva,121,31
8B · B1 fija,H2,H0,Repite H1,121,31
8B · B1 fija,H3,H0,Repite H1,121,31
8B · B2 adaptativa,H1,H0,Nueva,121,31


7 propuestas duplicadas detectadas; 28 llamadas de evaluación omitidas.


Siete duplicados evitaron 28 reevaluaciones. El modelo sigue repitiendo propuestas; no se midió diversidad semántica.

Longitud sin tokens especiales: $\ell_H=|Tokenizer(H)|$ y $\ell_\phi=|Tokenizer(\phi)|$. El consumo total incluye plantilla, contratos, datos y salidas. Longitud no mide claridad.


## 5. Decisión y mensaje

Valores originales del modelo. Un mensaje coherente puede expresar una decisión incorrecta.


In [5]:
record = next(r for r in records if r["report"]["model_key"] == "qwen3-8b" and r["protocol"]["strategy_mode"] == "fixed")
item = record["report"]["rounds"][1]
row = item["validation"]["rows"][0]
panel("Política H1 propuesta por 8B, rechazada", item["policy"])
panel("Contexto visible", json.dumps(row["context"], ensure_ascii=False, indent=2))
panel("Decisión devuelta por el LLM", json.dumps(row["raw_reply"], ensure_ascii=False, indent=2))
panel("Mensaje generado por la plantilla", row["reply"]["message"])
table(["Comprobación", "Resultado"], [
    ["Decisión esperada", json.dumps(row["expected"], ensure_ascii=False)],
    ["Coincide con la tarea", "Sí" if row["task_match"] else "No"],
    ["Incumplimientos detectados", ", ".join(row["violations"]) or "Ninguno"],
    ["Respuesta correcta", "Sí" if row["success"] else "No"],
])

Comprobación,Resultado
Decisión esperada,"{""action"": ""offer"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100}"
Coincide con la tarea,No
Incumplimientos detectados,Ninguno
Respuesta correcta,No


## 6. Casos reservados

Seis casos fijados antes de ejecutar y usados después de seleccionar. No llegan al mejorador ni al actualizador. Son escenarios sintéticos, no una muestra representativa de conversaciones.


In [6]:
rows = []
for index, case in enumerate(records[0]["protocol"]["cases"]["transfer"]):
    expected = case["expected"]
    plan = f'{expected["action"]} / {expected["debt"]} / {expected["payments"]} / {expected["installment"]}'
    outcomes = ["Correcta" if r["report"]["transfer"]["selected"]["rows"][index]["success"] else "Incorrecta" for r in records]
    rows.append([case["id"].replace("transfer-", ""), plan, *outcomes])
table(["Caso", "Esperado: acción/deuda/pagos/cuota", *[label(r) for r in records]], rows)

Caso,Esperado: acción/deuda/pagos/cuota,4B · B1 fija,4B · B2 adaptativa,8B · B1 fija,8B · B2 adaptativa
unknown,ask / None / 0 / 0,Incorrecta,Incorrecta,Correcta,Correcta
capacity-150,offer / B / 2 / 150,Incorrecta,Incorrecta,Incorrecta,Incorrecta
capacity-300,offer / C / 2 / 300,Incorrecta,Incorrecta,Incorrecta,Incorrecta
capacity-600,offer / C / 1 / 600,Incorrecta,Incorrecta,Incorrecta,Incorrecta
declined,stop / None / 0 / 0,Incorrecta,Incorrecta,Correcta,Correcta
insufficient,stop / None / 0 / 0,Incorrecta,Incorrecta,Incorrecta,Incorrecta


## 7. Política de referencia

Se sustituye H por una política manual fija $H_{ref}$. El mejorador y el actualizador no participan:

$$x \longrightarrow \mathrm{Run}(L,H_{ref},x) \longrightarrow \text{decisión} \longrightarrow \text{evaluador fijo}.$$

La entrada contiene solo historial, deudas y pagos permitidos. Las respuestas esperadas no se envían al modelo. Se mantiene el contrato de cuatro campos y el mensaje por plantilla, sin corregir decisiones.

**Criterio:** una respuesta acierta si coincide con acción, deuda, número de pagos y cuota esperados y no viola las reglas. Un JSON válido o la ausencia de incumplimientos no bastan.

In [7]:
panel("H_ref: política manual, idéntica para 4B y 8B", reference_records[0]["protocol"]["reference_policy"])
rows = []
for record in reference_records:
    r = record["report"]
    groups = r["reference"]["splits"]
    score = lambda group: f'{sum(row["success"] for row in group["rows"])}/{len(group["rows"])}'
    all_rows = [row for group in groups.values() for row in group["rows"]]
    rows.append(["8B" if r["model_key"] == "qwen3-8b" else "4B",
                 *[score(groups[s]) for s in ("development", "validation", "test", "transfer")],
                 f'{sum(row["success"] for row in all_rows)}/12',
                 sum(bool(row["violations"]) for row in all_rows)])
table(["Modelo", "Desarrollo", "Validación", "Test reutilizado", "Seis casos v5", "Total", "Con incumplimientos"], rows)


Modelo,Desarrollo,Validación,Test reutilizado,Seis casos v5,Total,Con incumplimientos
4B,2/2,0/2,2/2,3/6,7/12,4
8B,1/2,0/2,1/2,4/6,6/12,0


### Políticas de referencia

**H0:** inicio defectuoso, conservado por la búsqueda v5. **H_ref:** control manual independiente; sus aciertos no son automejora. Los grupos mantienen sus nombres, aunque ahora todos son conocidos.


In [8]:
rows = []
for record in reference_records:
    r = record["report"]
    previous = [item for item in records if item["report"]["model_key"] == r["model_key"]]
    base = previous[0]["report"]
    baseline = base["rounds"][0]
    best_candidate_q = max(item["validation"]["q"] for old in previous for item in old["report"]["rounds"][1:])
    groups = r["reference"]["splits"]
    score = lambda group: f'{sum(row["success"] for row in group["rows"])}/{len(group["rows"])}'
    rows.append(["8B" if r["model_key"] == "qwen3-8b" else "4B",
        score(baseline["validation"]), f'{best_candidate_q:.0%}', score(groups["validation"]),
        score(base["transfer"]["initial"]), score(groups["transfer"]),
        r["reference"]["policy_tokens"], r["tokens"], f'{r["elapsed_seconds"]:.1f}', f'{r["peak_mlx_gb"]:.2f}'])
table(["Modelo", "H0 validación", "Mejor Q candidato v5", "H_ref validación", "H0 seis casos", "H_ref seis casos", "Tokens H_ref", "Tokens totales", "Segundos", "Pico MLX GB"], rows)


Modelo,H0 validación,Mejor Q candidato v5,H_ref validación,H0 seis casos,H_ref seis casos,Tokens H_ref,Tokens totales,Segundos,Pico MLX GB
4B,0/2,0%,0/2,0/6,3/6,382,10085,12.6,3.20
8B,0/2,0%,0/2,2/6,4/6,382,10022,17.3,5.43


### Respuestas por caso

Formato: **acción / deuda / pagos / cuota**. `offer / C / 2 / 300`: C en dos cuotas de 300. Con capacidad 600, es viable pero falla la preferencia por un pago.


In [9]:
def decision_text(value):
    if value is None:
        return "Salida inválida"
    return " / ".join(str(value[k]) if value[k] is not None else "null"
                      for k in ("action", "debt", "payments", "installment"))

rows = []
for split in ("development", "validation", "test", "transfer"):
    left, right = [record["report"]["reference"]["splits"][split]["rows"] for record in reference_records]
    for a, b in zip(left, right):
        assert a["id"] == b["id"]
        def outcome(row):
            status = "Correcta" if row["success"] else "Incorrecta"
            reason = " · cuota excedida" if "exceeds_declared_capacity" in row["violations"] else ""
            return f'{decision_text(row["raw_reply"])} · {status}{reason}'
        rows.append([a["id"], decision_text(a["expected"]), outcome(a), outcome(b)])
table(["Caso", "Esperado", "4B", "8B"], rows)


Caso,Esperado,4B,8B
development-largest,offer / C / 2 / 300,offer / C / 2 / 300 · Correcta,ask / null / 0 / 0 · Incorrecta
development-declined,stop / null / 0 / 0,stop / null / 0 / 0 · Correcta,stop / null / 0 / 0 · Correcta
validation-limited,offer / A / 2 / 100,offer / C / 2 / 300 · Incorrecta · cuota excedida,stop / null / 0 / 0 · Incorrecta
validation-infeasible,stop / null / 0 / 0,offer / C / 2 / 300 · Incorrecta · cuota excedida,ask / null / 0 / 0 · Incorrecta
test-largest,offer / C / 2 / 300,offer / C / 2 / 300 · Correcta,ask / null / 0 / 0 · Incorrecta
test-consent,accept / B / 2 / 150,accept / B / 2 / 150 · Correcta,accept / B / 2 / 150 · Correcta
transfer-unknown,ask / null / 0 / 0,ask / null / 0 / 0 · Correcta,ask / null / 0 / 0 · Correcta
transfer-capacity-150,offer / B / 2 / 150,offer / C / 2 / 300 · Incorrecta · cuota excedida,offer / B / 2 / 150 · Correcta
transfer-capacity-300,offer / C / 2 / 300,offer / C / 2 / 300 · Correcta,offer / C / 2 / 300 · Correcta
transfer-capacity-600,offer / C / 1 / 600,offer / C / 2 / 300 · Incorrecta,offer / C / 2 / 300 · Incorrecta


## 8. Ejecuciones anteriores

Se conservan fallos y pruebas incompletas. V4 incorporó historial al feedback. V5 añadió plantilla de mensajes, memoria, duplicados y casos reservados.

Cambiaron varios componentes; no se aísla su efecto. El contraste 4B/8B comparte v5.


In [10]:
previous = []
for folder in sorted((case_dir / "results").glob("*-local-*")):
    if folder.name in run_names + reference_names:
        continue
    record = load_record(folder.name)
    r, p = record["report"], record["protocol"]
    previous.append([folder.name, p["protocol"].rsplit("-", 1)[-1],
        "Completa" if r["status"] == "completed" else "Detenida", len(r["rounds"]) - 1 if r["rounds"] else 0])
table(["Registro previo", "Protocolo", "Estado", "Propuestas evaluadas"], previous)

Registro previo,Protocolo,Estado,Propuestas evaluadas
20261031T052541Z-local-qwen,v1,Detenida,0
20261031T052617Z-local-qwen,v2,Detenida,2
20261031T052638Z-local-gemma,v2,Detenida,0
20261031T052812Z-local-qwen3,v2,Completa,3
20261031T053709Z-local-qwen3-calibration,v3,Completa,0
20261031T053730Z-local-qwen3-explicit-fixed,v3,Completa,3
20261031T053758Z-local-qwen3-explicit-adaptive,v3,Completa,3
20261031T053849Z-local-qwen3-explicit-fixed,v4,Completa,3
20261031T053935Z-local-qwen3-explicit-adaptive,v4,Completa,3


## 9. Diagnóstico del bloqueo

- **Negociador:** la política manual falla validación en ambos modelos. 4B excede capacidad; 8B pregunta o se detiene cuando corresponde otra acción.
- **Mejorador:** no obtiene una política aceptada. El control manual no aísla su contribución.
- **Escalado:** 8B mejora un acierto en el grupo de seis, pero pierde uno en el total. No justifica sustituir 4B en esta prueba.
- **Continuación:** comprobar capacidad, elección de deuda y cuota antes de retomar búsqueda.

Una ejecución por modelo, a 4 bits y sin razonamiento extendido. Política manual posterior a fallos conocidos; 12 casos reutilizados. Diagnóstico, no generalización. La longitud tampoco aísla contenido, orden o claridad.

## Repetir el control

Python 3.12 en Apple Silicon, [requirements-local.txt](../requirements-local.txt) y snapshots locales fijados:

```sh
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/local_model_test.py qwen3 --contract decision --reference
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/local_model_test.py qwen3-8b --contract decision --reference
```

Ejecutar uno a la vez. Máximo 12 llamadas, 50.000 tokens y 900 segundos por modelo. Sin búsqueda, reintentos, API ni entrenamiento. El modo de búsqueda v5 sigue disponible con `--strategy fixed` o `--strategy adaptive`, sin `--reference`.

Cada ejecución conserva protocolo, respuestas originales, métricas, script y hashes en `results/`. Este notebook muestra los registros fijados al inicio.

Fuentes: [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) y [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B). Las revisiones MLX utilizadas figuran en cada protocolo.
